# Image → volumetric 3D (Hunyuan3D-2) — Kaggle GPU

Turns each input **image** into a real, volumetric 3D **mesh** using
Hunyuan3D-2 shape generation — far better than the fast TripoSR path for
flat/illustration inputs, which TripoSR collapses to a flat sheet.

**Before running**
1. *Session options → Accelerator*: **GPU T4 x2** (sm_75). ⚠️ **Not P100** —
   Kaggle's PyTorch dropped Pascal (sm_60), so a P100 crashes on the first
   CUDA op. API-launched runs always get a P100; pick the T4 in the **web UI**
   and *Run All* (or *Save & Run All*). The install runs first so a P100 run
   still verifies the environment before the GPU guard stops it.
2. *Session options → Internet*: **On** (downloads the model + pip packages).
3. *Add Input* → attach your images as a Kaggle **Dataset** (1–20 images).

**Output:** `/kaggle/working/editable_scene.zip` with `objects/000.glb …`
and `labels.json` — import into the local app as a mesh scene.

In [ ]:
# ---------------- Config ----------------
from pathlib import Path
INPUT_DIR = Path("/kaggle/input")     # searched recursively for images
WORK = Path("/kaggle/working")
OUT = WORK / "out"; OBJ = OUT / "objects"
for d in (OUT, OBJ): d.mkdir(parents=True, exist_ok=True)

MAX_IMAGES = 20        # cap objects per scene
REMOVE_BG = True       # cut the object out of its background first
STEPS = 50             # DiT sampling steps (model default; 30 was under-cooked)
OCTREE_RES = 384       # shape-decoder grid (model default; 256 loses detail)
GUIDANCE = 5.0         # classifier-free guidance (model default)
PROJECT_COLOR = True   # best-effort: paint front-facing verts from the image

# The DiT is generative: some seeds collapse a hard 2D input to a flat sheet.
# Generate with several seeds, keep the most volumetric result, and stop early
# once one is clearly 3D. This is what turns "flat cutout" into a real object.
SEEDS = [0, 1, 2, 3]
GOOD_RATIO = 0.25      # thinnest/longest axis considered "properly 3D"

In [ ]:
%%bash
# ---------------- Install Hunyuan3D-2 shape gen (~2 min; no GPU needed) ----------------
# Installed BEFORE the GPU guard so a P100 run tests the whole cell (including
# the hy3dgen.shapegen import). Cloned to /tmp so kernel OUTPUT stays tiny.
#
# CRITICAL: Kaggle already ships torch/numpy/scipy/transformers/opencv. We add
# ONLY the missing deps and must NOT let pip upgrade numpy/scipy — doing so
# (via rembg/scikit-image) broke scipy<->numpy at import ("_blas_supports_fpe").
# `trimesh` needs only numpy (no scipy upgrade); diffusers/omegaconf/einops are
# pure/light. hy3dgen is imported via sys.path (no `pip install -e .`), and
# background removal uses a numpy threshold (no rembg) — nothing touches scipy.
set -e
cd /tmp
[ -d Hunyuan3D-2 ] || git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
echo ">>> clone OK"
# pymeshlab is a self-contained wheel (only needs numpy) that hy3dgen.shapegen
# imports at load time; the others are pure/light. None upgrade numpy/scipy.
pip -q install trimesh diffusers omegaconf einops pymeshlab || { echo ">>> deps FAILED"; exit 1; }
echo ">>> deps OK"
python - <<'PY'
import sys; sys.path.insert(0, "/tmp/Hunyuan3D-2")
import numpy, scipy, trimesh            # verify the numpy/scipy/trimesh stack is intact
print(">>> numpy", numpy.__version__, "| scipy", scipy.__version__, "| trimesh", trimesh.__version__)
import hy3dgen
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
print(">>> shapegen import OK")
PY

In [ ]:
# ---------------- GPU guard (fail fast on an unusable accelerator) ----------------
# Import the scientific stack IN THE KERNEL first — this is exactly what the
# model cell needs, and it reproduces the numpy/scipy break here (on any GPU)
# rather than after the model download.
import torch, numpy, scipy, trimesh
print(f"stack: numpy {numpy.__version__} | scipy {scipy.__version__} | "
      f"trimesh {trimesh.__version__} | torch {torch.__version__}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Session options -> Accelerator -> 'GPU T4 x2', then Run All.")
cap = torch.cuda.get_device_capability(); tag = f"sm_{cap[0]}{cap[1]}"
name = torch.cuda.get_device_name(0); arch = torch.cuda.get_arch_list()
print(f"GPU: {name} ({tag}) | torch {torch.__version__}")
print(f"torch supports: {arch}")
if tag not in arch:
    raise SystemExit(
        f"\n{'='*68}\nThis session got a {name} ({tag}), unsupported by this PyTorch.\n"
        f"Set Accelerator to 'GPU T4 x2' in the Kaggle UI, then Run All.\n"
        f"(API-launched runs always get a P100; only the UI can pick the T4.)\n{'='*68}"
    )
_ = (torch.zeros(16, device='cuda') + 1).sum().item()
print("GPU OK — install verified and a CUDA kernel launched.")

In [ ]:
# ---------------- Load model + collect images ----------------
import sys, glob, numpy as np
sys.path.insert(0, "/tmp/Hunyuan3D-2")   # hy3dgen imported from the clone
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

from scipy import ndimage  # ships with Kaggle; used for border flood-fill

def to_rgba(img, white_thresh=245):
    # Hunyuan3D uses the alpha channel as the foreground mask. Remove only the
    # white that is CONNECTED TO THE IMAGE BORDER (true background) — keep
    # interior white (e.g. white stripes ON the object) solid, so the model
    # sees a solid object, not a holey/striped cut-out (which collapses to a
    # flat sheet). If the image already has real alpha, trust it.
    if img.mode == 'RGBA' and np.asarray(img)[..., 3].min() < 250:
        return img
    rgb = np.asarray(img.convert('RGB'))
    white = (rgb > white_thresh).all(2)
    lbl, _ = ndimage.label(white)
    border = set(lbl[0]) | set(lbl[-1]) | set(lbl[:, 0]) | set(lbl[:, -1])
    border.discard(0)
    bg = np.isin(lbl, list(border)) if border else np.zeros_like(white)
    alpha = np.where(bg, 0, 255).astype('uint8')
    return Image.fromarray(np.dstack([rgb, alpha]), 'RGBA')

pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')

exts = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.PNG', '*.JPG', '*.JPEG')
paths = []
for e in exts:
    paths += glob.glob(str(INPUT_DIR / '**' / e), recursive=True)
paths = sorted(set(paths))[:MAX_IMAGES]
print(f"found {len(paths)} image(s):", [Path(p).name for p in paths])
assert paths, "No images found — Add Input -> attach an image dataset."

In [ ]:
# ---------------- Optional: bake front-view vertex colors from the image ----------------
def project_colors(mesh, pil_img):
    # Hunyuan3D returns a mesh normalized ~[-1,1] facing the camera (+Z).
    # Paint front-facing vertices by sampling the image at their (x,y).
    try:
        rgb = np.asarray(pil_img.convert('RGB'))
        H, W = rgb.shape[:2]
        v = np.asarray(mesh.vertices); n = np.asarray(mesh.vertex_normals)
        xy = v[:, :2].copy()
        mn, mx = xy.min(0), xy.max(0)
        uv = (xy - mn) / np.clip(mx - mn, 1e-6, None)     # 0..1
        u = np.clip((uv[:, 0] * (W - 1)).astype(int), 0, W - 1)
        vv = np.clip(((1 - uv[:, 1]) * (H - 1)).astype(int), 0, H - 1)
        cols = rgb[vv, u]
        front = n[:, 2] >= 0
        base = np.full_like(cols, 180)                    # gray for back faces
        base[front] = cols[front]
        mesh.visual = __import__('trimesh').visual.color.ColorVisuals(
            mesh, vertex_colors=np.concatenate([base, np.full((len(base), 1), 255, np.uint8)], 1))
    except Exception as e:
        print("colour projection skipped:", e)
    return mesh

In [ ]:
# ---------------- Generate a mesh per image ----------------
import json, torch, trimesh

def volume_ratio(m):
    """thinnest/longest bbox axis: ~0 = flat cutout, ~1 = chunky solid."""
    e = sorted(m.bounds[1] - m.bounds[0])
    return float(e[0] / max(e[2], 1e-6))

def save_model_input(img_in, path):
    # Save EXACTLY what the model is conditioned on (after recenter/compose).
    # Cheap, and it turns 'why is it flat?' into evidence instead of guesswork.
    try:
        from hy3dgen.shapegen.preprocessors import ImageProcessorV2
        out = ImageProcessorV2()(img_in, to_tensor=False)
        Image.fromarray(out['image']).save(path)
    except Exception as e:
        print("  (preview skipped:", e, ")")

labels = {}
for i, p in enumerate(paths):
    name = Path(p).stem
    print(f"[{i+1}/{len(paths)}] {name}", flush=True)
    img = Image.open(p).convert('RGB')
    img_in = to_rgba(img) if REMOVE_BG else img
    save_model_input(img_in, OUT / f"preview_{i:03d}_model_input.png")

    best, best_r = None, -1.0
    for s in SEEDS:
        gen = torch.Generator(device='cuda').manual_seed(int(s))
        try:
            m = pipe(image=img_in, num_inference_steps=STEPS,
                     octree_resolution=OCTREE_RES, guidance_scale=GUIDANCE,
                     generator=gen)[0]
        except TypeError:      # older/newer hy3dgen kwarg differences
            m = pipe(image=img_in)[0]
        r = volume_ratio(m)
        print(f"   seed {s}: verts={len(m.vertices):>6} ratio={r:.3f}", flush=True)
        if r > best_r:
            best, best_r = m, r
        if best_r >= GOOD_RATIO:
            break          # already properly 3D — don't spend more GPU time

    mesh = best
    if best_r < 0.10:
        print(f"   !! still flat (ratio {best_r:.3f}) — the input may be a hard "
              f"2D vector image; a shaded/photographic version reconstructs far better.")
    if PROJECT_COLOR:
        mesh = project_colors(mesh, img)
    ext = mesh.bounds[1] - mesh.bounds[0]
    print(f"  BEST verts={len(mesh.vertices)} bbox={np.round(ext,3)} ratio={best_r:.3f}")
    mesh.export(str(OBJ / f"{i:03d}.glb"))
    labels[str(i)] = name

(OUT / "labels.json").write_text(json.dumps(labels, indent=2))
print("wrote", len(labels), "objects to", OBJ)

In [ ]:
# ---------------- Package for the local app ----------------
import shutil
zip_path = shutil.make_archive(str(WORK / "editable_scene"), "zip", root_dir=OUT)
print("done ->", zip_path)
for f in sorted(OBJ.glob("*.glb")):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.2f} MB)")

## Troubleshooting

| Symptom | Fix |
|---|---|
| `SystemExit` at the GPU guard / `sm_60` / `no kernel image` | You're on a **P100**. Session options → Accelerator → **GPU T4 x2**, then Run All. |
| Out of memory | Lower `OCTREE_RES` to 192 and `MAX_IMAGES`; process fewer images per run. |
| Result still looks flat | Use a cleaner single-object image; very thin/2D inputs have little shape to infer. |
| `hy3dgen` import errors | Re-run the install cell once (first `import` can lazily compile small helpers). |